# LLM-Based Complaint Classification (with keyword baseline kept)

This notebook keeps the original keyword-rule approach and adds two simple LLM-based paths for comparison:
1. a local Ollama model
2. a Groq API model using the key from .env

What you will see here:
1. Keyword / regex baseline (unchanged)
2. Ollama-based classification
3. Groq-based classification
4. A side-by-side comparison on a small sample of complaints


In [11]:
import warnings
warnings.filterwarnings('ignore')

import json
import os
import re
import urllib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

plt.style.use('seaborn-v0_8-whitegrid')

load_dotenv()

DATA_PATH = Path('..') / 'clarity_bookings_dataset.csv'
df = pd.read_csv(DATA_PATH)

# Keep only rows with real complaint text
complaints_df = df[df['customer_complaint'].notna()].copy()
complaints_df['customer_complaint'] = complaints_df['customer_complaint'].astype(str)

print('Rows for complaint analysis:', complaints_df.shape[0])
print('Sample complaints:')
print(complaints_df['customer_complaint'].head(5).to_string(index=False))


Rows for complaint analysis: 248
Sample complaints:
             Meal preference not available onboard
                  Schedule change not communicated
             Meal preference not available onboard
Connection time too short, missed connecting fl...
Flight delayed by 3 hours, requesting compensation


In [12]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace('refund', 'refund')
    text = text.replace('cancel', 'cancel')
    text = text.replace('delay', 'delay')
    text = re.sub(r"can't", 'cannot', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"[^a-z0-9\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

complaints_df['clean_text'] = complaints_df['customer_complaint'].apply(clean_text)

keyword_rules = {
    'Refund Issues': [r'\brefund\b', r'\bchargeback\b', r'\breturn\b'],
    'Schedule Change': [r'\bdelay\b', r'\breschedul\b', r'\bchange flight\b', r'\btime change\b'],
    'Baggage': [r'\bbaggage\b', r'\blost luggage\b', r'\bdamaged bag\b'],
    'Pricing Error': [r'\bprice\b', r'\bfare\b', r'\bcharged\b', r'\bextra fee\b'],
    'Ticketing Issues': [r'\bticket\b', r'\bbooking error\b', r'\bconfirmation\b', r'\breservation\b'],
    'Customer Service': [r'\bagent\b', r'\bcustomer service\b', r'\bcall center\b'],
}


def rule_label(text: str) -> str:
    text = text.lower()
    for label, patterns in keyword_rules.items():
        if any(re.search(p, text) for p in patterns):
            return label
    return 'Other'

complaints_df['rule_category'] = complaints_df['clean_text'].apply(rule_label)

print('Keyword category distribution:')
print(complaints_df['rule_category'].value_counts().to_string())


Keyword category distribution:
rule_category
Other               129
Ticketing Issues     54
Baggage              31
Refund Issues        20
Pricing Error        14


In [13]:
OLLAMA_HOST = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'llama3:latest')


if 'clean_text' not in complaints_df.columns:
    complaints_df['clean_text'] = complaints_df['customer_complaint'].apply(clean_text)

if 'rule_category' not in complaints_df.columns:
    complaints_df['rule_category'] = complaints_df['clean_text'].apply(rule_label)


def classify_with_llm(text: str) -> dict:
    """Return an Ollama category for a complaint, with a safe fallback."""
    fallback_label = rule_label(text)

    prompt = f"""
You are classifying customer complaints for a travel booking company.
Choose exactly one category from this list:
Refund Issues, Schedule Change, Baggage, Pricing Error,
Ticketing Issues, Customer Service, Other.
Complaint text: "{text}"
Return only the category name.
"""

    try:
        payload = {
            'model': MODEL_NAME,
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': 0}
        }
        req = urllib.request.Request(
            f'{OLLAMA_HOST}/api/generate',
            data=json.dumps(payload).encode('utf-8'),
            headers={'Content-Type': 'application/json'},
            method='POST'
        )
        with urllib.request.urlopen(req, timeout=120) as response:
            result = json.loads(response.read().decode('utf-8'))

        category = result.get('response', '').strip().splitlines()[-1].strip()
        if not category:
            raise ValueError('Empty Ollama response.')

        return {'category': category, 'note': 'Ollama classification succeeded.'}
    except Exception as exc:
        return {
            'category': fallback_label,
            'note': f'Ollama call failed ({type(exc).__name__}): using keyword fallback.'
        }


sample_df = complaints_df[['customer_complaint', 'clean_text', 'rule_category']].head(10).copy()
sample_df['llm_category'] = sample_df['clean_text'].apply(lambda x: classify_with_llm(x)['category'])

sample_df['llm_note'] = sample_df['clean_text'].apply(lambda x: classify_with_llm(x)['note'])

print('LLM sample results (first 10 complaints):')
print(sample_df[['customer_complaint', 'rule_category', 'llm_category', 'llm_note']].to_string(index=False))


LLM sample results (first 10 complaints):
                                 customer_complaint rule_category    llm_category                         llm_note
              Meal preference not available onboard         Other         Baggage Ollama classification succeeded.
                   Schedule change not communicated         Other Schedule Change Ollama classification succeeded.
              Meal preference not available onboard         Other         Baggage Ollama classification succeeded.
Connection time too short, missed connecting flight         Other Schedule Change Ollama classification succeeded.
 Flight delayed by 3 hours, requesting compensation         Other Schedule Change Ollama classification succeeded.
                 Refund not processed after 30 days Refund Issues   Refund Issues Ollama classification succeeded.
   Duplicate charge on credit card for same booking         Other   Pricing Error Ollama classification succeeded.
         Unable to add extra baggage t

In [14]:
from groq import Groq

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.1-8b-instant')


def classify_with_groq(text: str) -> dict:
    """Return a Groq-based category for a complaint, with a safe fallback."""
    fallback_label = rule_label(text)

    if not GROQ_API_KEY:
        return {
            'category': fallback_label,
            'note': 'Groq API key not found in .env. Using keyword fallback.'
        }

    try:
        client = Groq(api_key=GROQ_API_KEY)
        prompt = f"""
You are classifying customer complaints for a travel booking company.
Choose exactly one category from this list:
Refund Issues, Schedule Change, Baggage, Pricing Error,
Ticketing Issues, Customer Service, Other.
Complaint text: "{text}"
Return only the category name.
"""

        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {'role': 'system', 'content': 'You are a careful complaint categorizer.'},
                {'role': 'user', 'content': prompt}
            ],
            temperature=0,
            max_tokens=30,
        )

        category = response.choices[0].message.content.strip().splitlines()[-1].strip()
        if not category:
            raise ValueError('Empty Groq response.')

        return {'category': category, 'note': 'Groq classification succeeded.'}
    except Exception as exc:
        return {
            'category': fallback_label,
            'note': f'Groq call failed ({type(exc).__name__}): using keyword fallback.'
        }


sample_df = complaints_df[['customer_complaint', 'clean_text', 'rule_category']].head(10).copy()
sample_df['groq_category'] = sample_df['clean_text'].apply(lambda x: classify_with_groq(x)['category'])
sample_df['groq_note'] = sample_df['clean_text'].apply(lambda x: classify_with_groq(x)['note'])

print('Groq sample results (first 10 complaints):')
print(sample_df[['customer_complaint', 'rule_category', 'groq_category', 'groq_note']].to_string(index=False))


Groq sample results (first 10 complaints):
                                 customer_complaint rule_category   groq_category                      groq_note
              Meal preference not available onboard         Other   Pricing Error Groq classification succeeded.
                   Schedule change not communicated         Other Schedule Change Groq classification succeeded.
              Meal preference not available onboard         Other   Pricing Error Groq classification succeeded.
Connection time too short, missed connecting flight         Other Schedule Change Groq classification succeeded.
 Flight delayed by 3 hours, requesting compensation         Other Schedule Change Groq classification succeeded.
                 Refund not processed after 30 days Refund Issues   Refund Issues Groq classification succeeded.
   Duplicate charge on credit card for same booking         Other   Pricing Error Groq classification succeeded.
         Unable to add extra baggage through portal  